In [26]:
import os
import requests
import pandas as pd
import numpy as np

from dotenv import load_dotenv

load_dotenv()

google_maps_key = os.getenv("GOOGLE_MAPS_API_KEY")

print("API key configured:", google_maps_key is not None)

API key configured: True


 Reusable Nearby Competitor Collector

A reusable function is created to retrieve nearby businesses for a proposed location and business type.

The function:
- queries Google Places Nearby Search
- retrieves relevant place information
- calculates distance from the proposed location
- removes duplicate place IDs
- returns a clean DataFrame

The configured radius is currently 3 km for the prototype.

In [27]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two geographic coordinates
    using the Haversine formula.

    Returns distance in kilometers.
    """

    R = 6371  # Earth's radius in km

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = lat2 - lat1
    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [28]:
def get_nearby_competitors(
    latitude,
    longitude,
    business_type,
    radius_km=3,
    max_results=20
):
    url = "https://places.googleapis.com/v1/places:searchNearby"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": google_maps_key,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.formattedAddress,"
            "places.rating,"
            "places.userRatingCount,"
            "places.priceLevel,"
            "places.types,"
            "places.location"
        )
    }

    payload = {
        "includedTypes": [business_type],
        "maxResultCount": max_results,
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": latitude,
                    "longitude": longitude
                },
                "radius": radius_km * 1000
            }
        }
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        raise Exception(
            f"Places API error {response.status_code}: "
            f"{response.text}"
        )

    data = response.json()

    rows = []

    for place in data.get("places", []):
        rows.append({
            "place_id": place.get("id"),
            "name": place.get("displayName", {}).get("text"),
            "address": place.get("formattedAddress"),
            "rating": place.get("rating"),
            "review_count": place.get("userRatingCount"),
            "price_level": place.get("priceLevel"),
            "types": place.get("types"),
            "latitude": place.get("location", {}).get("latitude"),
            "longitude": place.get("location", {}).get("longitude")
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    # Remove duplicate businesses
    df = df.drop_duplicates(
        subset="place_id"
    ).reset_index(drop=True)

    # Calculate distance from proposed location
    df["distance_km"] = haversine_distance(
        latitude,
        longitude,
        df["latitude"],
        df["longitude"]
    )

    # Keep only businesses inside requested radius
    df = df[
        df["distance_km"] <= radius_km
    ].copy()

    return df.reset_index(drop=True)

In [29]:
import joblib

rf_model = joblib.load("../models/zomato_performance_model.pkl")
preprocessor = joblib.load("../models/zomato_preprocessor.pkl")

print("Model loaded:", type(rf_model).__name__)
print("Preprocessor loaded:", type(preprocessor).__name__)

Model loaded: RandomForestClassifier
Preprocessor loaded: ColumnTransformer


In [30]:
test_business = pd.DataFrame([{
    "online_order": 1,
    "book_table": 0,
    "approx_costfor_two_people": 500,
    "log_cost": np.log1p(500),
    "cost_band": "Mid-Range",
    "location": "Whitefield",
    "primary_cuisine": "Cafe",
    "cuisine_count": 3,
    "primary_rest_type": "Cafe",
    "historical_restaurant_count": 1231,
    "location_median_cost": 400,
    "location_online_order_rate": 0.70,
    "location_book_table_rate": 0.10,
    "location_cuisine_diversity": 40,
    "location_business_type_diversity": 15
}])

display(test_business)

,online_order,book_table,approx_costfor_two_people,log_cost,cost_band,location,primary_cuisine,cuisine_count,primary_rest_type,historical_restaurant_count,location_median_cost,location_online_order_rate,location_book_table_rate,location_cuisine_diversity,location_business_type_diversity
0,1,0,500,6.216606,Mid-Range,Whitefield,Cafe,3,Cafe,1231,400,0.7,0.1,40,15


In [31]:
test_business_encoded = preprocessor.transform(test_business)

print("Encoded shape:", test_business_encoded.shape)

Encoded shape: (1, 219)


In [32]:
prediction = rf_model.predict(test_business_encoded)

probabilities = rf_model.predict_proba(
    test_business_encoded
)

print("Prediction:", prediction[0])

print("\nProbabilities:")

for class_name, probability in zip(
    rf_model.classes_,
    probabilities[0]
):
    print(f"{class_name}: {probability:.3f}")

Prediction: Low

Probabilities:
High: 0.295
Low: 0.436
Medium: 0.270


In [33]:
def analyze_business_opportunity(
    probabilities,
    model_classes,
    competition_summary
):
    """
    Combine historical ML performance and current competition
    into a rule-based business opportunity assessment.
    """

  
    # 1. Historical performance

    
    class_probabilities = dict(
        zip(model_classes, probabilities)
    )

    high_probability = class_probabilities.get("High", 0)
    medium_probability = class_probabilities.get("Medium", 0)

    performance_score = (
        high_probability * 100
        + medium_probability * 50
    )

   
    # 2. Competition strength

    competitor_count = competition_summary[
        "competitor_count_retrieved"
    ]

    avg_rating = competition_summary[
        "avg_competitor_rating"
    ]

    avg_reviews = competition_summary[
        "avg_competitor_reviews"
    ]

    high_rating_count = competition_summary[
        "high_rating_competitors"
    ]

    high_review_count = competition_summary[
        "high_review_competitors"
    ]

    # Avoid division by zero
    if competitor_count > 0:

        competition_score = (
            (avg_rating / 5) * 40
            +
            min(avg_reviews / 2000, 1) * 30
            +
            (high_rating_count / competitor_count) * 15
            +
            (high_review_count / competitor_count) * 15
        )

    else:
        competition_score = 0

   
    # 3. Opportunity score
   

    opportunity_score = (
        0.60 * performance_score
        +
        0.40 * (100 - competition_score)
    )

   
    # 4. Signals


    if performance_score < 40:
        performance_signal = "Weak historical performance"
    elif performance_score < 60:
        performance_signal = "Moderate historical performance"
    else:
        performance_signal = "Strong historical performance"

    if competition_score >= 70:
        competition_signal = "Strong competition"
    elif competition_score >= 40:
        competition_signal = "Moderate competition"
    else:
        competition_signal = "Limited competition"

    
    # 5. Opportunity classification
   

    if opportunity_score < 40:
        opportunity_class = "Low"
    elif opportunity_score < 70:
        opportunity_class = "Moderate"
    else:
        opportunity_class = "High"

    
    # 6. Final structured output
   

    return {
        "opportunity_score": round(
            float(opportunity_score), 2
        ),
        "opportunity_class": opportunity_class,

        "historical_performance": round(
            float(performance_score), 2
        ),

        "competition_strength": round(
            float(competition_score), 2
        ),

        "performance_signal": performance_signal,
        "competition_signal": competition_signal
    }